# Обучение ИИ-сегментатора текста для манги
Этот ноутбук предназначен для запуска в **Google Colab (с подключенным T4 GPU)**.
Он автоматически сгенерирует качественный синтетический датасет (скринтоны, линии, текст с обводкой) и обучит нейросеть U-Net пиксельной сегментации текста.
В конце обучения модель будет экспортирована в формат `segmenter.onnx` для MangaEditor.

In [ ]:
# 1. Установка необходимых библиотек в Google Colab
!pip install segmentation-models-pytorch albumentations onnx onnxruntime opencv-python

In [ ]:
# 2. Импорт библиотек
import os
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

In [ ]:
# 3. Генератор синтетических данных (скринтоны + случайные линии + текст с обводкой)
class SyntheticMangaDataset(Dataset):
    def __init__(self, size=1000, transform=None):
        self.size = size
        self.transform = transform
        # Набор случайных символов (японские кандзи, хирагана, катакана, английские буквы)
        self.characters = "あいうえおかきくけこさしすせсоたちつてとなにぬねのはひふへほまみむめもяゆよらりるれろわをん" \
                          "アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲン" \
                          "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789!?" \
                          "猫の国空飛ぶ船魔法使い冒険剣と魔法世界の果て" # Кандзи
        
    def __len__(self):
        return self.size
        
    def generate_screentone(self, width=256, height=256):
        # Генерируем искусственный растр (точки скринтона)
        bg = np.full((height, width), random.randint(220, 255), dtype=np.uint8)
        if random.random() > 0.4:
            # Рисуем регулярную сетку точек скринтона
            dot_dist = random.randint(4, 12)
            dot_size = random.randint(1, 3)
            dot_color = random.randint(80, 180)
            # Смещение фазы
            ox, oy = random.randint(0, 10), random.randint(0, 10)
            for y in range(oy, height, dot_dist):
                for x in range(ox, width, dot_dist):
                    cv2.circle(bg, (x, y), dot_size, dot_color, -1)
        return bg
        
    def draw_random_lines(self, img):
        # Рисуем случайные линии рисунка (чтобы модель училась НЕ сегментировать их)
        num_lines = random.randint(0, 5)
        for _ in range(num_lines):
            p1 = (random.randint(0, 256), random.randint(0, 256))
            p2 = (random.randint(0, 256), random.randint(0, 256))
            color = random.randint(0, 100)
            thickness = random.randint(1, 4)
            cv2.line(img, p1, p2, color, thickness)
            
    def __getitem__(self, idx):
        # 1. Генерируем фон манги
        img = self.generate_screentone(256, 256)
        self.draw_random_lines(img)
        
        # Добавляем случайное размытие фона
        if random.random() > 0.5:
            img = cv2.GaussianBlur(img, (3, 3), 0)
            
        # 2. Рисуем текст и создаем маску
        mask = np.zeros((256, 256), dtype=np.uint8)
        
        # Решаем, сколько блоков текста рисовать (от 1 до 4)
        num_texts = random.randint(1, 4)
        for _ in range(num_texts):
            # Случайное слово
            text_len = random.randint(2, 10)
            text = "".join(random.choice(self.characters) for _ in range(text_len))
            
            # Случайный шрифт (в Colab по умолчанию есть стандартные шрифты Hershey)
            font = random.choice([
                cv2.FONT_HERSHEY_SIMPLEX,
                cv2.FONT_HERSHEY_DUPLEX,
                cv2.FONT_HERSHEY_COMPLEX
            ])
            scale = random.uniform(0.6, 1.3)
            thickness = random.randint(1, 3)
            
            # Нам нужны размеры текста для позиционирования
            text_size = cv2.getTextSize(text, font, scale, thickness)[0]
            tx = random.randint(0, max(1, 256 - text_size[0]))
            ty = random.randint(text_size[1], 256 - 5)
            
            # Рисуем обводку текста (белую) на изображении
            outline_color = random.randint(220, 255)
            cv2.putText(img, text, (tx, ty), font, scale, outline_color, thickness + 4, cv2.LINE_AA)
            
            # Рисуем сами черные буквы на изображении
            text_color = random.randint(0, 30)
            cv2.putText(img, text, (tx, ty), font, scale, text_color, thickness, cv2.LINE_AA)
            
            # Записываем точную маску текста (буквы + обводка)
            cv2.putText(mask, text, (tx, ty), font, scale, 255, thickness + 2, cv2.LINE_AA)
            
        # Нормализуем изображения
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            # Добавляем размерность каналов вручную
            img = torch.tensor(img).unsqueeze(0)
            mask = torch.tensor(mask).unsqueeze(0)
            
        return img, mask

In [ ]:
# 4. Аугментации для тренировки
train_transform = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.GaussNoise(var_limit=(0.001, 0.01), p=0.3),
    ToTensorV2()
])

val_transform = A.Compose([
    ToTensorV2()
])

# Создаем датасеты
train_dataset = SyntheticMangaDataset(size=5000, transform=train_transform)
val_dataset = SyntheticMangaDataset(size=500, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
# 5. Определение модели U-Net с бэкбоном ResNet34
# Вход: 1 канал (серый), Выход: 1 канал (маска)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

model = smp.Unet(
    encoder_name="resnet34",        # бэкбон ResNet34
    encoder_weights="imagenet",     # предобученные веса ImageNet
    in_channels=1,                  # 1 входной канал (серое изображение)
    classes=1,                      # 1 выходной канал (логиты маски)
    activation=None
)
model = model.to(device)

In [ ]:
# 6. Функции потерь и оптимизатор
criterion = smp.losses.DiceLoss(mode="binary", from_logits=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

In [ ]:
# 7. Тренировочный цикл (15 эпох на GPU займет около 10-15 минут)
num_epochs = 15
best_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for imgs, masks in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{num_epochs}"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * imgs.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Валидация
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * imgs.size(0)
            
    val_loss /= len(val_loader.dataset)
    scheduler.step()
    
    print(f"Эпоха {epoch+1} завершена. Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_manga_segmenter.pth")
        print("-> Сохранены новые лучшие веса!")

In [ ]:
# 8. Экспорт модели в формат ONNX для MangaEditor
model.load_state_dict(torch.load("best_manga_segmenter.pth", map_location="cpu"))
model.eval()

# Создаем тестовый тензор входа под форму (1, 1, 256, 256)
dummy_input = torch.randn(1, 1, 256, 256, dtype=torch.float32)

onnx_path = "segmenter.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],            # имя входа как в MangaEditor
    output_names=['output'],          # имя выхода
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)
print(f"Модель успешно экспортирована в {onnx_path}!")
print("Скачайте файл segmenter.onnx и положите его в папку models вашего проекта!")